# Praat Scripts (日本語版) - ピッチ・F0統計量の一括抽出 (Google Colab版) 🎙️

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okawawaka/praat-scripts-ja/blob/main/notebooks/collect_pitch_data.ipynb)

本ノートブックは、`scripts/acoustic-analysis/collect_pitch_data.praat` と**全く同一のアルゴリズム（To Pitch）**で、各区間のピッチ（基本周波数 F0）の平均値・中央値・最小値・最大値・標準偏差を一括測定するツールです。

Praat の C++ コアエンジン（`praat-parselmouth`）を使用しているため、**Praat本体で測定した場合と100%同一のピッチ統計データ**が得られます。

### Step 1: 必要なライブラリのインストール

In [ ]:
!pip install -q praat-parselmouth textgrid pandas

### Step 2: 分析パラメータの設定
話者の声の高さに合わせてピッチ探索範囲（下限・上限）を設定してください。

In [ ]:
# @title 【パラメータ設定】 { run: "auto" }
target_tier = 1 # @param {type:"integer"}
pitch_floor_hz = 75 # @param {type:"number"}
pitch_ceiling_hz = 500 # @param {type:"number"}
skip_empty_labels = True # @param {type:"boolean"}

print(f"設定完了: Tier={target_tier}, F0探索範囲={pitch_floor_hz}〜{pitch_ceiling_hz} Hz")

### Step 3: 音声ファイル(.wav)とTextGrid(.TextGrid)をアップロード
下のセルを実行して表示される「ファイル選択」ボタンから、**分析したいペア（.wav と .TextGrid）をまとめて選択**してください（複数ペア可）。

In [ ]:
import os
import tempfile
import parselmouth
from parselmouth.praat import call
import textgrid
import pandas as pd
from google.colab import files

print("▼ 下の『ファイル選択』ボタンをクリックして、.wav と .TextGrid ファイルをまとめて選択してください：")
uploaded = files.upload()

if not uploaded:
    print("ファイルが選択されませんでした。")
else:
    temp_dir = tempfile.mkdtemp()
    for fname, data in uploaded.items():
        with open(os.path.join(temp_dir, fname), "wb") as f:
            f.write(data)

    all_rows = []
    wav_files = [f for f in uploaded.keys() if f.lower().endswith(".wav")]

    for w_name in wav_files:
        base = os.path.splitext(w_name)[0]
        tg_candidates = [base + ".TextGrid", base + ".textgrid"]
        tg_name = next((c for c in tg_candidates if c in uploaded), None)

        if not tg_name:
            print(f"スキップ: {w_name} に対応する TextGrid が見つかりません。")
            continue

        wav_path = os.path.join(temp_dir, w_name)
        tg_path = os.path.join(temp_dir, tg_name)

        try:
            sound = parselmouth.Sound(wav_path)
            pitch = sound.to_pitch(time_step=0.01, pitch_floor=pitch_floor_hz, pitch_ceiling=pitch_ceiling_hz)

            tg = textgrid.TextGrid.fromFile(tg_path)
            if target_tier <= len(tg.tiers):
                tier = tg.tiers[target_tier - 1]
                if isinstance(tier, textgrid.IntervalTier):
                    for idx, interval in enumerate(tier, start=1):
                        label = interval.mark.strip()
                        if skip_empty_labels and label == "":
                            continue

                        start_t = interval.minTime
                        end_t = interval.maxTime
                        duration_ms = (end_t - start_t) * 1000.0

                        # PraatのTo Pitchオブジェクトから各区間の統計量を直接クエリ
                        mean_f0 = call(pitch, "Get mean", start_t, end_t, "Hertz")
                        med_f0 = call(pitch, "Get quantile", start_t, end_t, 0.5, "Hertz")
                        min_f0 = call(pitch, "Get minimum", start_t, end_t, "Hertz", "Parabolic")
                        max_f0 = call(pitch, "Get maximum", start_t, end_t, "Hertz", "Parabolic")
                        std_f0 = call(pitch, "Get standard deviation", start_t, end_t, "Hertz")

                        def fmt(v):
                            return f"{v:.2f}" if v is not None and not pd.isna(v) else "NA"

                        all_rows.append({
                            "Filename": base,
                            "IntervalIndex": idx,
                            "Label": label,
                            "StartTime_s": f"{start_t:.4f}",
                            "EndTime_s": f"{end_t:.4f}",
                            "Duration_ms": f"{duration_ms:.2f}",
                            "MeanPitch_Hz": fmt(mean_f0),
                            "MedianPitch_Hz": fmt(med_f0),
                            "MinPitch_Hz": fmt(min_f0),
                            "MaxPitch_Hz": fmt(max_f0),
                            "StDevPitch_Hz": fmt(std_f0)
                        })
        except Exception as e:
            print(f"エラー ({base}): {e}")

    if all_rows:
        df_pitch = pd.DataFrame(all_rows)
        print(f"\n集計完了: 合計 {len(df_pitch)} 件のピッチ統計データを抽出しました。")
    else:
        df_pitch = pd.DataFrame()
        print("\n集計対象のデータがありませんでした。")

### Step 4: 抽出結果の確認（Praatと完全同一のテーブル形式）

In [ ]:
if 'df_pitch' in locals() and not df_pitch.empty:
    display(df_pitch)
else:
    print("表示するデータがありません。")

### Step 5: TSVファイルのダウンロード

In [ ]:
if 'df_pitch' in locals() and not df_pitch.empty:
    out_tsv = "pitch_results.tsv"
    df_pitch.to_csv(out_tsv, sep="\t", index=False, encoding="utf-8-sig")
    print(f"{out_tsv} をダウンロードします...")
    files.download(out_tsv)
else:
    print("ダウンロード対象のデータがありません。")